# Parsing DIMACS CNF with DeepLog


This short notebook shows how to load DIMACS CNF text, map it onto DeepLog formulas with `parse_dimacs_cnf`, and choose the target structure (boolean vs. probability/logprobability).

## Setup

In [ ]:
from torch.backends.cudnn import deterministic

from deeplog.formula import parse_dimacs_cnf, SymbolicFormulaFactory, parse_formula
from deeplog.formula.deeplogmodulefactory import DeepLogModuleFactory
import torch

## 1. Describe the CNF
We'll use a tiny CNF with two clauses over three literals: 
$ (x_1 \lor \lnot x_3) \land (x_2 \lor x_3 \lor \lnot x_1) $.

In [ ]:
dimacs_text = """
c Example CNF
p cnf 3 2
1 -3 0
2 3 -1 0
"""
print(dimacs_text)


## 2. Parse to a symbolic formula (boolean structure)
By default, the parser emits boolean surface operations (`and`, `or`, `not`).

In [ ]:
symbolic_boolean = parse_dimacs_cnf(
    dimacs_text, SymbolicFormulaFactory(), structure="boolean"
)
symbolic_boolean

The parser prefixes variable ids with `v` (e.g., `v1`, `v2`) to avoid colliding with neutral constants like `1`/`0` that are already reserved inside DeepLog circuits.

## 3. Target a different structure
Pass `structure='probability'` (or `logprobability`) to immediately map the CNF onto the semiring-compatible operators (`times`/`plus`/`negate`).

In [ ]:
symbolic_prob = parse_dimacs_cnf(
    dimacs_text, SymbolicFormulaFactory(), structure="probability"
)
symbolic_prob

## 4. Compile to a module
The same parser can feed a `DeepLogModuleFactory` to obtain a `DeepLogModule` that evaluates the CNF under weighted semantics. Because the factory remaps boolean surface operators internally, we keep `structure='boolean'` while building the formula.

The resulting module expects probabilities for each atom and returns the overall CNF probability.


In [ ]:
from deeplog import reshape
from deeplog.shape import SymTensor

module_factory = DeepLogModuleFactory()
wmc_formula = parse_dimacs_cnf(dimacs_text, module_factory, structure="probability")
wmc_formula.root.circuit.deterministic = True
wmc_module = wmc_formula.to_module(("cnf",))
print("Original input shape:", wmc_module.get_input_shape())

# Fix input ordering to v1, v2, v3
expected_input = SymTensor([
    ("_", ("v1",), ("probability",)),
    ("_", ("v2",), ("probability",)),
    ("_", ("v3",), ("probability",)),
])
wmc_module = reshape(wmc_module, input=expected_input)
print("Input shape:", wmc_module.get_input_shape())

# Evaluate: P(v1)=0.8, P(v2)=0.35, P(v3)=0.6
inputs = torch.tensor([[0.8, 0.35, 0.6]])
wmc_module(inputs)
